In [1]:
from dotenv import load_dotenv
import os

from huggingface_hub import login

from pricer.evaluator import evaluate
from pricer.deep_neural_network import DeepNeuralNetworkRunner
from pricer.items import Item

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)

True
2.10.0+cu126


In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA built: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu126
CUDA built: 12.6
CUDA available: True


In [5]:
import sys
print(sys.executable)

c:\Users\cloud\Documents\projects\master-llm-eng\code\.venv\Scripts\python.exe


In [4]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

In [6]:
# Load dataset

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation_items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation_items, 10,000 test items


In [7]:
runner = DeepNeuralNetworkRunner(train, val[:1000])
runner.setup()

Deep Neural Network created with 289,128,449 parameters
Using cuda


In [9]:
runner.train(epochs=5)

  0%|          | 0/12500 [00:00<?, ?it/s]

Epoch [1/5]
Train Loss: 0.5395, Val Loss: 0.4365
Val mean absolute error: $59.22
Learning rate: 0.001000


  0%|          | 0/12500 [00:00<?, ?it/s]

Epoch [2/5]
Train Loss: 0.3835, Val Loss: 0.4169
Val mean absolute error: $57.27
Learning rate: 0.000976


  0%|          | 0/12500 [00:00<?, ?it/s]

Epoch [3/5]
Train Loss: 0.3235, Val Loss: 0.4050
Val mean absolute error: $54.52
Learning rate: 0.000905


  0%|          | 0/12500 [00:00<?, ?it/s]

Epoch [4/5]
Train Loss: 0.2813, Val Loss: 0.4007
Val mean absolute error: $54.65
Learning rate: 0.000794


  0%|          | 0/12500 [00:00<?, ?it/s]

Epoch [5/5]
Train Loss: 0.2453, Val Loss: 0.4016
Val mean absolute error: $54.71
Learning rate: 0.000655


In [11]:
def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$7 $52 $17 $80 $32 $71 $55 $16 $11 $17 $18 $173 $15 $32 $11 $6 $32 $22 $1 $41 $13 $104 $7 $100 $79 $121 $195 $1 $42 $57 $36 $2 $136 $47 $7 $66 $29 $12 $60 $6 $128 $15 $14 $2 $135 $5 $16 $3 $70 $12 $5 $27 $214 $39 $17 $13 $13 $92 $45 $11 $163 $47 $17 $38 $297 $0 $14 $222 $25 $79 $18 $5 $3 $42 $6 $14 $28 $1 $11 $3 $68 $30 $8 $68 $10 $40 $139 $116 $48 $89 $2 $18 $1 $1 $8 $103 $9 $8 $62 $218 $2 $18 $2 $100 $7 $53 $23 $271 $2 $47 $31 $45 $6 $41 $5 $21 $14 $13 $31 $147 $5 $79 $43 $54 $93 $70 $1 $20 $30 $36 $3 $21 $11 $4 $62 $2 $7 $12 $34 $10 $5 $166 $64 $65 $18 $6 $9 $70 $59 $10 $2 $23 $2 $29 $11 $19 $80 $1 $29 $11 $21 $15 $12 $6 $121 $1 $23 $20 $3 $13 $45 $4 $272 $35 $33 $7 $5 $0 $47 $27 $19 $1 $62 $41 $7 $10 $63 $21 $23 $9 $5 $2 $9 $99 $29 $33 $22 $8 $28 $13 

In [13]:
import plotly.graph_objects as go

results = [
    ("Random Pricer", "gray", 382.08),
    ("Gemma 270B", "slateblue", 202.10),
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("Human", "black", 87.62),
    ("NLP + LR (BoW)", "gray", 76.81),
    ("Random Forest", "gray", 73.04),
    ("GPT-4.1-nano FT", "skyblue", 68.91),
    ("GPT-4.1-nano", "slateblue", 68.29),
    ("XGBoost", "gray", 68.23),
    ("Vanilla 8L NN", "orange", 58.82),
    ("Optimized 3L NN", "orange", 51.53),
    ("Claude Opus 4.6", "slateblue", 49.14),
    ("GPT-5.1", "slateblue", 48.24),
    ("10L Residual NN", "orange", 40.74),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction Error by Model (Worst → Best)",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800,
)

fig.show()